# Week 12 - Activity 1: Synthetic Data Generation

In this activity, we'll explore generating synthetic data using LLMs:
1. Generate diverse examples for a specific task
2. Evaluate data quality and diversity
3. Compare with real data distributions
4. Identify potential biases and limitations

We'll use this to understand the strengths and weaknesses of synthetic data.

## Setup and Imports

In [2]:
import os
import json
import pandas as pd
import numpy as np
from litellm import completion
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Any
from tqdm.notebook import tqdm

## Helper Functions

First, let's define some helper functions for JSON parsing and data analysis.

In [3]:
def extract_json_from_text(text: str) -> str:
    """Extract JSON content from text that might contain markdown code blocks."""
    # Look for JSON content between triple backticks
    if "```json" in text:
        start = text.find("```json") + 7
        end = text.find("```", start)
        if start > 6 and end > start:
            return text[start:end].strip()
    
    # If no code block found, look for array directly
    start = text.find("[{")
    end = text.rfind("}") + 1
    if start >= 0 and end > start:
        return text[start:end].strip()
    
    return text

## Data Generation

Now let's define the function that generates synthetic examples using the LLM.

In [4]:
def generate_examples(task_description: str, n_examples: int = 10) -> List[Dict]:
    """Generate synthetic examples for a given task."""
    examples = []
    
    prompt = f"""
    Generate {n_examples} diverse examples for the following task:
    {task_description}
    
    Return the examples in JSON format, with each example containing:
    - input: the task input
    - output: the expected output
    - metadata: any relevant information about the example
    
    Ensure the examples are diverse and cover different cases.
    """
    
    print("\nSending request to LLM...")
    
    response = completion(
        model=os.getenv('LLM_MODEL'),
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        api_base=os.getenv('LLM_BASE_URL'),
        api_key=os.getenv('LLM_API_KEY')
    )
    
    print("\nReceived response from LLM. Processing...")
    response_content = response.choices[0].message.content
    print("\nRaw response:", response_content)
    
    # Extract and parse JSON
    try:
        json_content = extract_json_from_text(response_content)
        print("\nExtracted JSON content:", json_content)
        examples = json.loads(json_content)
        print(f"\nSuccessfully parsed JSON. Found {len(examples)} examples.")
    except json.JSONDecodeError as e:
        print(f"\nError parsing JSON: {e}")
        # Return a minimal valid example to allow the rest of the code to run
        examples = [
            {'input': 'This is a fallback example', 'output': 'positive'},
            {'input': 'Another fallback example', 'output': 'negative'}
        ]
    
    return examples

## Data Quality Evaluation

Let's define functions to evaluate the quality of our synthetic data.

In [5]:
def evaluate_data_quality(examples: List[Dict]) -> Dict[str, float]:
    """Evaluate various aspects of data quality."""
    # Extract text content
    texts = [ex['input'] for ex in examples]
    print(f"\nProcessing {len(texts)} texts for quality evaluation")
    print("Sample texts:", texts[:2])
    
    # Calculate diversity metrics
    vectorizer = TfidfVectorizer(stop_words=None, min_df=1)  # Modified to handle small datasets
    try:
        tfidf_matrix = vectorizer.fit_transform(texts)
        pairwise_sim = cosine_similarity(tfidf_matrix)
        
        # Average similarity between examples
        avg_similarity = (np.sum(pairwise_sim) - len(texts)) / (len(texts) * (len(texts) - 1))
        
        # Vocabulary richness
        vocab_size = len(vectorizer.vocabulary_)
        avg_length = np.mean([len(text.split()) for text in texts])
        
        # Label distribution
        labels = [ex['output'] for ex in examples]
        label_counts = pd.Series(labels).value_counts(normalize=True)
        label_entropy = -np.sum(label_counts * np.log2(label_counts))
        
        return {
            'avg_similarity': avg_similarity,
            'vocab_size': vocab_size,
            'avg_length': avg_length,
            'label_entropy': label_entropy
        }
    except Exception as e:
        print(f"\nError in evaluate_data_quality: {e}")
        return {
            'avg_similarity': 0.0,
            'vocab_size': 0,
            'avg_length': 0.0,
            'label_entropy': 0.0
        }

## Distribution Analysis

Let's analyze the distributions of various features in our synthetic data.

In [6]:
def analyze_distributions(examples: List[Dict]) -> Dict[str, Any]:
    """Analyze various distributions in the dataset."""
    if not examples:
        return {
            'lengths': [],
            'sentiments': [],
            'word_freq': pd.Series(),
            'style_features': {
                'emoji_rate': 0.0,
                'punctuation_rate': 0.0
            }
        }
    
    texts = [ex['input'] for ex in examples]
    
    # Text length distribution
    lengths = [len(text.split()) for text in texts]
    
    # Sentiment distribution
    sentiments = [ex['output'] for ex in examples]
    
    # Vocabulary distribution
    words = [word.lower() for text in texts for word in text.split()]
    word_freq = pd.Series(words).value_counts()
    
    # Writing style features
    has_emoji = [any(char in text for char in '😊😂🤔😍😢') for text in texts]
    has_punctuation = [any(char in text for char in '!?...') for text in texts]
    
    return {
        'lengths': lengths,
        'sentiments': sentiments,
        'word_freq': word_freq,
        'style_features': {
            'emoji_rate': sum(has_emoji) / len(texts),
            'punctuation_rate': sum(has_punctuation) / len(texts)
        }
    }

## Bias Analysis

Let's check for potential biases in our synthetic data.

In [7]:
def analyze_biases(examples: List[Dict]) -> Dict[str, Any]:
    """Analyze potential biases in the dataset."""
    texts = [ex['input'].lower() for ex in examples]
    
    # Define categories to check
    categories = {
        'gender': ['he', 'she', 'his', 'her', 'man', 'woman'],
        'age': ['young', 'old', 'elderly', 'teen', 'adult'],
        'occupation': ['doctor', 'nurse', 'engineer', 'teacher', 'worker'],
        'ethnicity': ['asian', 'black', 'white', 'hispanic', 'african'],
        'location': ['urban', 'rural', 'city', 'country', 'suburban']
    }
    
    bias_metrics = {}
    for category, terms in categories.items():
        # Count occurrences of each term
        term_counts = {}
        for term in terms:
            count = sum(1 for text in texts if term in text.split())
            term_counts[term] = count
        
        # Calculate distribution metrics
        total = sum(term_counts.values())
        if total > 0:
            distribution = {k: v/total for k, v in term_counts.items()}
            # Calculate entropy as a measure of balance
            entropy = -sum(p * np.log2(p) for p in distribution.values() if p > 0)
        else:
            distribution = {k: 0 for k in term_counts.keys()}
            entropy = 0
            
        bias_metrics[category] = {
            'distribution': distribution,
            'entropy': entropy,
            'total_mentions': total
        }
    
    return bias_metrics

## Visualization Functions

Let's define functions to visualize our analysis results using seaborn.

In [8]:
def plot_quality_comparison(synthetic_metrics: Dict[str, float], real_metrics: Dict[str, float]):
    """Plot data quality comparison using seaborn."""
    # Convert metrics to DataFrame for plotting
    metrics_df = pd.DataFrame({
        'Metric': list(synthetic_metrics.keys()),
        'Synthetic': list(synthetic_metrics.values()),
        'Real': list(real_metrics.values())
    })
    
    # Melt DataFrame for seaborn
    metrics_df_melted = pd.melt(metrics_df, id_vars=['Metric'], var_name='Dataset', value_name='Value')
    
    # Create bar plot
    plt.figure(figsize=(10, 6))
    sns.barplot(data=metrics_df_melted, x='Metric', y='Value', hue='Dataset')
    plt.title('Data Quality Metrics Comparison')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

def plot_distributions(synthetic_dist: Dict[str, Any], real_dist: Dict[str, Any]):
    """Plot distribution analysis using seaborn."""
    # Text length distribution
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    sns.histplot(data=pd.DataFrame({
        'Length': synthetic_dist['lengths'] + real_dist['lengths'],
        'Dataset': ['Synthetic'] * len(synthetic_dist['lengths']) + ['Real'] * len(real_dist['lengths'])
    }), x='Length', hue='Dataset', multiple="layer", alpha=0.5)
    plt.title('Text Length Distribution')
    
    plt.subplot(1, 2, 2)
    sentiment_counts = pd.DataFrame({
        'Synthetic': pd.Series(synthetic_dist['sentiments']).value_counts(),
        'Real': pd.Series(real_dist['sentiments']).value_counts()
    }).fillna(0)
    sns.barplot(data=sentiment_counts)
    plt.title('Sentiment Distribution')
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    plt.show()

def plot_bias_analysis(synthetic_biases: Dict[str, Any], real_biases: Dict[str, Any]):
    """Plot bias analysis using seaborn."""
    # Create DataFrame for entropy comparison
    entropy_df = pd.DataFrame({
        'Category': list(synthetic_biases.keys()),
        'Synthetic': [d['entropy'] for d in synthetic_biases.values()],
        'Real': [d['entropy'] for d in real_biases.values()]
    })
    
    # Melt DataFrame for seaborn
    entropy_df_melted = pd.melt(entropy_df, id_vars=['Category'], var_name='Dataset', value_name='Entropy')
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=entropy_df_melted, x='Category', y='Entropy', hue='Dataset')
    plt.title('Category Distribution Entropy (higher = more balanced)')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## Main Execution

Now let's run all the analysis steps and see the results.

In [ ]:
# Set seaborn style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = [10, 6]

# Example task: Sentiment analysis
sentiment_task = """
Task: Generate examples for sentiment analysis.
Each example should be a text review and its sentiment (positive/negative/neutral).
Include various domains (products, services, movies, restaurants, etc.)
and different writing styles (formal, casual, with emojis, etc.).
"""

# Generate synthetic examples
synthetic_examples = generate_examples(sentiment_task, n_examples=10)
print("Generated examples:")
for i, example in enumerate(synthetic_examples, 1):
    print(f"\nExample {i}:")
    print(json.dumps(example, indent=2))

# Load real data for comparison
real_data = load_dataset('sst2', split='train').select(range(100))
real_examples = [
    {'input': item['sentence'], 'output': 'positive' if item['label'] == 1 else 'negative'}
    for item in real_data
]

# Evaluate data quality
synthetic_metrics = evaluate_data_quality(synthetic_examples)
real_metrics = evaluate_data_quality(real_examples)

# Compare metrics
comparison_df = pd.DataFrame({
    'Synthetic': synthetic_metrics,
    'Real': real_metrics
}).round(3)

print("\nData Quality Comparison:")
print(comparison_df)

# Plot quality comparison
plot_quality_comparison(synthetic_metrics, real_metrics)

# Analyze distributions
print("\nAnalyzing distributions...")
synthetic_dist = analyze_distributions(synthetic_examples)
real_dist = analyze_distributions(real_examples)

if synthetic_dist['lengths']:
    print("\nSynthetic Data Statistics:")
    print(f"Average text length: {np.mean(synthetic_dist['lengths']):.1f} words")
    print(f"Emoji usage rate: {synthetic_dist['style_features']['emoji_rate']*100:.1f}%")
    print(f"Punctuation usage rate: {synthetic_dist['style_features']['punctuation_rate']*100:.1f}%")
    print("\nSentiment distribution:")
    print(pd.Series(synthetic_dist['sentiments']).value_counts())
    
    # Plot distributions
    plot_distributions(synthetic_dist, real_dist)

# Analyze biases
synthetic_biases = analyze_biases(synthetic_examples)
real_biases = analyze_biases(real_examples)

print("\nBias Analysis:")
for category in synthetic_biases.keys():
    print(f"\n{category.title()} Category:")
    print(f"Synthetic entropy: {synthetic_biases[category]['entropy']:.3f}")
    print(f"Real entropy: {real_biases[category]['entropy']:.3f}")
    print("\nTerm distribution (Synthetic):")
    print(json.dumps(synthetic_biases[category]['distribution'], indent=2))

# Plot bias analysis
plot_bias_analysis(synthetic_biases, real_biases)